In [ ]:
%reset -f
%matplotlib inline
%config InlineBackend.figure_format='retina'
import math
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def gossip_coverage_with_dedup(N, d, L):
    """
    Returns the expectation of the total number of nodes reached by the last hop, where each node drops gossips
    already seen (i.e., not forwarded if seen earlier, called gossip deduplication).
    Also returns the network packet load in the last hop.
    
    Assuming perfect deduplication (every seen gossip is dropped); imperfection slightly increases reach and
    the forwaring load but the difference is insignificant.
    
    Assuming each node picks a random peer per transmit; implying assumption of reasonably uniform peer sampling.

    N: number of nodes in the network
    d: unicast gossip outdegree (number of destinations per hop)
    L: gossip message TTL (forward hop count)
    """
    S = 1  # informed nodes (nodes that received the gossip), incl. the origin
    A = 1  # expansion per hop (nodes that became informed in that hop), initially just the origin node
    for hop in range(L):
        M = d * A  # number of transmissions across all nodes at this hop
        # Model spread A_h+1 as balls into bins / coupon's collector problem:
        # - https://en.wikipedia.org/wiki/Balls_into_bins_problem
        # - https://en.wikipedia.org/wiki/Coupon_collector%27s_problem
        A_next = (N - S) * (1 - np.exp(-M / (N-1)))
        S_next = S + A_next
        S, A = S_next, A_next
        #print(f"hop {hop}: reached {S: 4.0f}/{N} nodes so far, contacted {A: 3.0f} new nodes this hop")
    return int(S), int(d * A)


def gossip_coverage_no_dedup(N, d, L):
    """
    The counterpart for the coverage-with-dedup model that assumes that nodes do not deduplicate seen gossips.
    Simple algebraic closed forms exist, but we use this iterative model for parity to the dedup model.
    """
    S = 1        # unique informed nodes, includes origin
    copies = 1   # copies at current frontier (origin copy initially)
    M_last = 0
    for hop in range(L):
        M = d * copies                   # all copies forward
        M_last = M
        A_next = (N - S) * (1.0 - np.exp(-M / (N - 1)))  # see the other model
        S += A_next
        copies = M                       # next hop has one copy per tx
    return int(S), int(M_last)


#gossip_coverage_with_dedup(300, 2, 10)
gossip_coverage_no_dedup(300, 2, 10)

In [ ]:
Ns = [50, 300, 1000]
ds = [1, 2, 3, 4]
Ls = list(range(1, 17))
#model = gossip_coverage_with_dedup
model = gossip_coverage_no_dedup

fig, axes = plt.subplots(len(Ns), 1, figsize=(10, 7), sharex=True)

if len(Ns) == 1:
    axes = [axes]

cov_axes = []

for ax_load, N in zip(axes, Ns):
    ax_cov = ax_load.twinx()
    cov_axes.append(ax_cov)
    for d in ds:
        results = [model(N, d, L) for L in Ls]
        coverage = [r[0] for r in results]
        load = [r[1] for r in results]
        ax_load.plot(
            Ls,
            load,
            linestyle="--",
            marker="x",
            linewidth=1,
            label=f"load msg/hop, d={d}",
        )
        ax_cov.plot(
            Ls,
            coverage,
            marker="+",
            linewidth=2,
            label=f"nodes reached, d={d}",
        )

    ax_load.set_title(f"Network with N={N} nodes")
    ax_load.set_ylabel("d·A (msg/hop) [capped]")
    ax_cov.set_ylabel("S (total nodes informed)")
    ax_load.set_ylim(0, N*2)
    ax_cov.set_ylim(0, N)
    ax_load.set_xticks(Ls)
    ax_load.grid(True, alpha=0.3)

axes[-1].set_xlabel("L (gossip forwarding hops)")

handles_load, labels_load = axes[0].get_legend_handles_labels()
handles_cov, labels_cov = cov_axes[0].get_legend_handles_labels()
fig.legend(
    handles_load + handles_cov,
    labels_load + labels_cov,
    loc="center right",
)
fig.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()